# 03 - Generative VAE: Pretrain, Finetune, and Visualization

This notebook demonstrates a lightweight two-phase VAE workflow: a short pretraining pass (ZINC-style), followed by a finetune pass on target-like data. For demo purposes this runs on synthetic data and produces an RDKit grid image of example molecules.

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn torch-geometric rdkit-pypi tensorboard biopython

# Setup and imports
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
import numpy as np
from core.vae_architecture import GraphVAE

def get_working_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1).to('cuda')
            return torch.device('cuda')
        except Exception:
            pass
    if torch.backends.mps.is_available():
        try:
            torch.zeros(1).to('mps')
            return torch.device('mps')
        except Exception:
            pass
    return torch.device('cpu')

def normalize_edge_index(data):
    edge_index = getattr(data, 'edge_index', None)
    if edge_index is None:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if not hasattr(edge_index, 'ndim') or edge_index.ndim != 2:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if 0 in edge_index.shape:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if edge_index.shape[0] == 2:
        return data
    if edge_index.shape[1] == 2:
        data.edge_index = edge_index.t().contiguous()
        return data
    data.edge_index = torch.zeros((2, 0), dtype=torch.long)
    return data

def truncate_graph_edges(data, max_nodes):
    edge_index = getattr(data, 'edge_index', None)
    edge_attr = getattr(data, 'edge_attr', None)
    if edge_index is None or not hasattr(edge_index, 'ndim') or edge_index.ndim != 2 or edge_index.shape[1] == 0:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        if edge_attr is not None:
            data.edge_attr = edge_attr.new_zeros((0, edge_attr.shape[-1]))
        return data
    if edge_index.shape[0] == 2:
        node_mask = (edge_index[0] < max_nodes) & (edge_index[1] < max_nodes)
    else:
        node_mask = (edge_index[:, 0] < max_nodes) & (edge_index[:, 1] < max_nodes)
        edge_index = edge_index.t().contiguous()
    edge_index = edge_index[:, node_mask]
    if edge_attr is not None and hasattr(edge_attr, 'shape') and edge_attr.shape[0] == node_mask.shape[0]:
        edge_attr = edge_attr[node_mask]
    if edge_index.numel() == 0:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        if edge_attr is not None:
            data.edge_attr = edge_attr.new_zeros((0, edge_attr.shape[-1]))
        return data
    data.edge_index = edge_index
    if edge_attr is not None:
        data.edge_attr = edge_attr
    return data
OUT = ROOT / 'results' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
device = get_working_device()
print(f'Ready; device={device}')

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


Note: you may need to restart the kernel to use updated packages.


c:\Users\u2251865\.conda\envs\tree4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready; device=cuda


In [ ]:
# ===== PHASE 1: VAE PRETRAINING ON REAL ZINC DATA =====

DATA_PROC = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'

print('='*60)
print('PHASE 1: Pre-training on ZINC (Real Large-Scale Data)')
print('='*60)

# Load SMILES from ZINC (either .smi or .csv)
zinc_smiles = []
zinc_file_smi = DATA_RAW / 'zinc250k.smi'
zinc_file_csv = DATA_RAW / 'zinc250k.csv'

if zinc_file_smi.exists():
    print(f'\n✓ Loading ZINC from {zinc_file_smi.name}...')
    with open(zinc_file_smi, 'r') as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            if parts:
                zinc_smiles.append(parts[0])
    print(f'  Loaded {len(zinc_smiles)} SMILES')
elif zinc_file_csv.exists():
    print(f'\n✓ Loading ZINC from {zinc_file_csv.name}...')
    import pandas as pd
    zinc_df = pd.read_csv(zinc_file_csv)
    zinc_smiles = zinc_df['smiles'].astype(str).tolist()
    print(f'  Loaded {len(zinc_smiles)} SMILES')
else:
    print('⚠ ZINC file not found; using synthetic data fallback')
    zinc_smiles = None

# Slice to 30,000 molecules for pretraining
if zinc_smiles:
    n_pretrain = min(200000, len(zinc_smiles))
    zinc_smiles = zinc_smiles[:n_pretrain]
    print(f'✓ Using {len(zinc_smiles)} ZINC molecules for pretraining')
else:
    print('  (Pretraining phase will skip real data)')

# Import graph conversion function (copy from Notebook 01)
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen
from torch_geometric.data import Data as PyGData

def normalize_edge_index(data):
    edge_index = getattr(data, 'edge_index', None)
    if edge_index is None:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if not hasattr(edge_index, 'ndim') or edge_index.ndim != 2:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if 0 in edge_index.shape:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if edge_index.shape[0] == 2:
        return data
    if edge_index.shape[1] == 2:
        data.edge_index = edge_index.t().contiguous()
        return data
    data.edge_index = torch.zeros((2, 0), dtype=torch.long)
    return data

def get_node_features(atom):
    partial_charge = 0.0
    if atom.HasProp('_GasteigerCharge'):
        try:
            partial_charge = float(atom.GetProp('_GasteigerCharge'))
        except Exception:
            partial_charge = 0.0
    return [
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()),
        int(atom.GetIsAromatic()),
        atom.GetMass(),
        atom.GetTotalNumHs(),
        partial_charge
    ]

def get_edge_features(bond):
    bond_type = bond.GetBondType()
    return [
        int(bond_type == Chem.rdchem.BondType.SINGLE),
        int(bond_type == Chem.rdchem.BondType.DOUBLE),
        int(bond_type == Chem.rdchem.BondType.TRIPLE),
        int(bond_type == Chem.rdchem.BondType.AROMATIC)
    ]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        AllChem.ComputeGasteigerCharges(mol)
    except Exception:
        pass
    x = [get_node_features(atom) for atom in mol.GetAtoms()]
    edge_index = [[], []]
    edge_attr = []
    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index[0].extend([a1, a2])
        edge_index[1].extend([a2, a1])
        features = get_edge_features(bond)
        edge_attr.extend([features, features])
    data = PyGData(
        x=torch.tensor(x, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float)
    )
    return data

# Convert ZINC SMILES to PyG graphs
print('\nConverting ZINC SMILES to PyG graphs...')
zinc_graphs = []
if zinc_smiles:
    for i, smiles in enumerate(zinc_smiles):
        if i % 5000 == 0:
            print(f'  Processed {i}/{len(zinc_smiles)}')
        g = smiles_to_graph(smiles)
        if g is not None:
            zinc_graphs.append(g)
    zinc_graphs = [normalize_edge_index(g) for g in zinc_graphs]
    print(f'✓ Converted {len(zinc_graphs)} valid ZINC graphs')
else:
    print('  (Skipping graph conversion—using synthetic fallback)')

# Initialize VAE
max_nodes = 50
epochs_pre = 15

vae = GraphVAE(
    node_features=8,
    edge_features=4,  # Now using full 4-dimensional edge features
    hidden_dim=128,
    latent_dim=64,
    max_nodes=max_nodes,
    use_pocket_conditioning=False,
 )

# Move model to device
vae = vae.to(device)

# Ensure optimizer exists for pretraining
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

criterion_recon = torch.nn.MSELoss()

# Training loop on real ZINC data
print(f'\n✓ Training VAE pretraining phase')
print(f'  Graphs: {len(zinc_graphs)}, Epochs: {epochs_pre}, Max nodes: {max_nodes}')
print(f'  (Processing in mini-batches of 32 graphs)\n')

vae.train()

if len(zinc_graphs) > 0:
    batch_size = 32
    n_batches = (len(zinc_graphs) + batch_size - 1) // batch_size
    
    for ep in range(epochs_pre):
        total_loss = 0.0
        n_processed = 0
        
        # Process in mini-batches
        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(zinc_graphs))
            batch_graphs = zinc_graphs[start_idx:end_idx]
            
            if len(batch_graphs) == 0:
                continue
            
            optimizer.zero_grad()
            
            # Pad graphs to max_nodes and collect into batch
            batch_x_list = []
            batch_edge_index_list = []
            batch_edge_attr_list = []
            batch_indices = []
            node_offset = 0
            
            for g_idx, g in enumerate(batch_graphs):
                x = g.x  # (n_atoms, 6)
                n_atoms = x.shape[0]
                
                # Pad to max_nodes
                if n_atoms < max_nodes:
                    x = torch.cat([x, torch.zeros(max_nodes - n_atoms, x.shape[1], device=x.device, dtype=x.dtype)], dim=0)
                else:
                    x = x[:max_nodes]
                
                batch_x_list.append(x)
                
                # Truncate edges to the active node window before batching
                g = truncate_graph_edges(g, max_nodes)
                if g.edge_index.shape[1] > 0:
                    edge_idx = g.edge_index + node_offset
                    batch_edge_index_list.append(edge_idx)
                    batch_edge_attr_list.append(g.edge_attr)
                
                batch_indices.extend([g_idx] * max_nodes)
                node_offset += max_nodes
            
            X_batch = torch.cat(batch_x_list, dim=0)
            
            if batch_edge_index_list:
                edge_index_batch = torch.cat(batch_edge_index_list, dim=1)
                edge_attr_batch = torch.cat(batch_edge_attr_list, dim=0)
            else:
                edge_index_batch = torch.zeros((2, 0), dtype=torch.long)
                edge_attr_batch = torch.zeros((0, 4), dtype=torch.float)
            
            batch_tensor = torch.tensor(batch_indices, dtype=torch.long)
            global_feat = torch.zeros(len(batch_graphs), 2)
            
            # Move batch tensors to device
            X_batch = X_batch.to(device)
            edge_index_batch = edge_index_batch.to(device)
            edge_attr_batch = edge_attr_batch.to(device)
            batch_tensor = batch_tensor.to(device)
            global_feat = global_feat.to(device)

            # Forward pass
            mu, logvar = vae.encode(X_batch, edge_index_batch, edge_attr_batch, batch_tensor, global_feat)
            z = vae.reparameterize(mu, logvar)
            node_logits, edge_adj, edge_type = vae.decode(z, pocket_embedding=None)
            
            # Loss
            X_reshaped = X_batch.view(len(batch_graphs), max_nodes, -1)
            loss_recon = criterion_recon(node_logits, X_reshaped)
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            
            beta = 0.01 * (ep + 1) / epochs_pre
            loss = loss_recon + beta * kld
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item() * len(batch_graphs)
            n_processed += len(batch_graphs)
        
        avg_loss = total_loss / max(n_processed, 1)
        print(f'Epoch {ep+1}/{epochs_pre} — Loss: {avg_loss:.4f}, β: {beta:.4f}')
else:
    print('⚠ No ZINC graphs available; skipping pretraining')

# Save pretrained model
models_dir = ROOT / 'models'
models_dir.mkdir(exist_ok=True, parents=True)
torch.save(vae.state_dict(), models_dir / 'zinc_pretrained_graphvae.pth')
print(f'\n✓ Pretrained VAE saved: {models_dir / "zinc_pretrained_graphvae.pth"}')

PHASE 1: Pre-training on ZINC (Real Large-Scale Data)

✓ Loading ZINC from zinc250k.csv...
  Loaded 249455 SMILES
✓ Using 200000 ZINC molecules for pretraining

Converting ZINC SMILES to PyG graphs...
  Processed 0/200000
  Processed 5000/200000
  Processed 10000/200000
  Processed 15000/200000
  Processed 20000/200000
  Processed 25000/200000
  Processed 30000/200000
  Processed 35000/200000
  Processed 40000/200000
  Processed 45000/200000
  Processed 50000/200000
  Processed 55000/200000
  Processed 60000/200000
  Processed 65000/200000
  Processed 70000/200000
  Processed 75000/200000
  Processed 80000/200000
  Processed 85000/200000
  Processed 90000/200000
  Processed 95000/200000
  Processed 100000/200000
  Processed 105000/200000
  Processed 110000/200000
  Processed 115000/200000
  Processed 120000/200000
  Processed 125000/200000
  Processed 130000/200000
  Processed 135000/200000
  Processed 140000/200000
  Processed 145000/200000
  Processed 150000/200000
  Processed 155000

In [18]:
# ===== PHASE 2: FINE-TUNING ON REAL EGFR DATA WITH POCKET CONDITIONING =====

print('\n' + '='*60)
print('PHASE 2: Fine-tuning on EGFR (With Pocket Conditioning)')
print('='*60)

device = get_working_device()
print(f'Using device for fine-tuning: {device}')

def normalize_edge_index(data):
    edge_index = getattr(data, 'edge_index', None)
    if edge_index is None:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if not hasattr(edge_index, 'ndim') or edge_index.ndim != 2:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if 0 in edge_index.shape:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if edge_index.shape[0] == 2:
        return data
    if edge_index.shape[1] == 2:
        data.edge_index = edge_index.t().contiguous()
        return data
    data.edge_index = torch.zeros((2, 0), dtype=torch.long)
    return data

def load_pretrained_weights_with_padding(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, weights_only=False, map_location='cpu')
    model_state = model.state_dict()
    adapted_state = {}

    for key, value in checkpoint.items():
        if key not in model_state:
            continue

        target = model_state[key]
        if value.shape == target.shape:
            adapted_state[key] = value
            continue

        if key == 'encoder.node_encoder.weight' and value.ndim == 2 and target.ndim == 2 and target.shape[1] >= value.shape[1]:
            padded = target.clone()
            padded.zero_()
            padded[:, :value.shape[1]] = value
            adapted_state[key] = padded
            continue

        if key == 'decoder.mlp_expand.0.weight' and value.ndim == 2 and target.ndim == 2 and target.shape[1] >= value.shape[1]:
            padded = target.clone()
            padded.zero_()
            padded[:, :value.shape[1]] = value
            adapted_state[key] = padded
            continue

        if key == 'decoder.node_decoder.3.weight' and value.ndim == 2 and target.ndim == 2 and target.shape[0] >= value.shape[0]:
            padded = target.clone()
            padded.zero_()
            padded[:value.shape[0], :] = value
            adapted_state[key] = padded
            continue

        if key == 'decoder.node_decoder.3.bias' and value.ndim == 1 and target.ndim == 1 and target.shape[0] >= value.shape[0]:
            padded = target.clone()
            padded.zero_()
            padded[:value.shape[0]] = value
            adapted_state[key] = padded
            continue

    missing, unexpected = model.load_state_dict(adapted_state, strict=False)
    print(f'  Loaded {len(adapted_state)} tensors from checkpoint')
    if missing:
        print(f'  Missing keys initialized randomly: {len(missing)}')
    if unexpected:
        print(f'  Unexpected keys ignored: {len(unexpected)}')

# Recreate the VAE for a clean fine-tuning run
vae = GraphVAE(
    node_features=8,
    edge_features=4,
    hidden_dim=128,
    latent_dim=64,
    max_nodes=50,
    use_pocket_conditioning=True,
)
vae = vae.to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

# Load pretrained model
load_pretrained_weights_with_padding(vae, models_dir / 'zinc_pretrained_graphvae.pth')
print('\n✓ Loaded pretrained VAE weights')

# Load real EGFR graphs from Notebook 01
graph_file = DATA_PROC / 'graph_data.pt'
if graph_file.exists():
    egfr_graphs = torch.load(str(graph_file), weights_only=False)
    egfr_graphs = [normalize_edge_index(g) for g in egfr_graphs]
    print(f'✓ Loaded {len(egfr_graphs)} real EGFR graphs from {graph_file}')
else:
    print('⚠ EGFR graphs not found; skipping fine-tuning')
    egfr_graphs = []

# Extract REAL pocket embedding from T790M EGFR PDB structure
from core.pocket_extractor import get_pocket_embedding

pdb_file = DATA_RAW / 'pdb' / '3W2S.pdb'
if pdb_file.exists():
    print(f'\n✓ Extracting real T790M EGFR pocket from {pdb_file}')
    real_pocket_array = get_pocket_embedding(
        str(pdb_file), 
        ligand_code='LIG',  # Standard ligand code in 3W2S
        pocket_radius=7.0,  # 7Angstrom radius (matches our extraction logic)
        embedding_dim=128
    )
    print(f'  Pocket embedding shape: {real_pocket_array.shape}')
    print(f'  Pocket embedding stats: mean={real_pocket_array.mean():.4f}, std={real_pocket_array.std():.4f}')
    
    # Convert to tensor and replicate across all graphs (same pocket for all conditioned molecules)
    pocket_embedding_tensor = torch.tensor(real_pocket_array, dtype=torch.float32).unsqueeze(0)
    pocket_embeddings = pocket_embedding_tensor.repeat(len(egfr_graphs), 1)
    print(f'  Replicated to all {len(egfr_graphs)} graphs')
else:
    print(f'⚠ PDB file not found at {pdb_file}; using fallback synthetic embeddings')
    # Fallback (only if PDB missing): create synthetic embeddings
    pocket_embeddings = torch.randn(len(egfr_graphs), 128, dtype=torch.float32) * 0.1

# Fine-tuning parameters
epochs_ft = 30
batch_size = 16

print(f'\n✓ Fine-tuning VAE with pocket conditioning')
print(f'  Graphs: {len(egfr_graphs)}, Epochs: {epochs_ft}, Batch size: {batch_size}')
print(f'  Pocket embedding: 128D (real T790M EGFR)\n')

if len(egfr_graphs) > 0:
    vae.train()
    n_batches = (len(egfr_graphs) + batch_size - 1) // batch_size
    
    for ep in range(epochs_ft):
        total_loss = 0.0
        n_processed = 0
        
        # Process in mini-batches
        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(egfr_graphs))
            batch_graphs = egfr_graphs[start_idx:end_idx]
            batch_pockets = pocket_embeddings[start_idx:end_idx]
            
            if len(batch_graphs) == 0:
                continue
            
            optimizer.zero_grad()
            
            # Batch processing (similar to Phase 1)
            batch_x_list = []
            batch_edge_index_list = []
            batch_edge_attr_list = []
            batch_indices = []
            node_offset = 0
            
            for g_idx, g in enumerate(batch_graphs):
                x = g.x
                n_atoms = x.shape[0]
                
                # Pad to max_nodes
                if n_atoms < max_nodes:
                    x = torch.cat([x, torch.zeros(max_nodes - n_atoms, x.shape[1], device=x.device, dtype=x.dtype)], dim=0)
                else:
                    x = x[:max_nodes]
                
                batch_x_list.append(x)
                
                # Truncate edges to the active node window before batching
                g = truncate_graph_edges(g, max_nodes)
                if g.edge_index.shape[1] > 0:
                    edge_idx = g.edge_index + node_offset
                    batch_edge_index_list.append(edge_idx)
                    batch_edge_attr_list.append(g.edge_attr)
                
                batch_indices.extend([g_idx] * max_nodes)
                node_offset += max_nodes
            
            X_batch = torch.cat(batch_x_list, dim=0)
            
            if batch_edge_index_list:
                edge_index_batch = torch.cat(batch_edge_index_list, dim=1)
                edge_attr_batch = torch.cat(batch_edge_attr_list, dim=0)
            else:
                edge_index_batch = torch.zeros((2, 0), dtype=torch.long)
                edge_attr_batch = torch.zeros((0, 4), dtype=torch.float)
            
            batch_tensor = torch.tensor(batch_indices, dtype=torch.long)
            global_feat = torch.zeros(len(batch_graphs), 2)
            pocket_batch = batch_pockets
            
            # Move batch tensors to device
            X_batch = X_batch.to(device)
            edge_index_batch = edge_index_batch.to(device)
            edge_attr_batch = edge_attr_batch.to(device)
            batch_tensor = batch_tensor.to(device)
            global_feat = global_feat.to(device)
            pocket_batch = pocket_batch.to(device)

            # Forward pass with pocket conditioning
            mu, logvar = vae.encode(X_batch, edge_index_batch, edge_attr_batch, batch_tensor, global_feat)
            z = vae.reparameterize(mu, logvar)
            node_logits, edge_adj, edge_type = vae.decode(z, pocket_embedding=pocket_batch)
            
            # Loss
            X_reshaped = X_batch.view(len(batch_graphs), max_nodes, -1)
            loss_recon = criterion_recon(node_logits, X_reshaped)
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            
            beta = 0.05 * (ep + 1) / epochs_ft  # Higher beta (fine-tuning regularization)
            loss = loss_recon + beta * kld
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item() * len(batch_graphs)
            n_processed += len(batch_graphs)
        
        avg_loss = total_loss / max(n_processed, 1)
        print(f'Epoch {ep+1}/{epochs_ft} — Loss: {avg_loss:.4f}, β: {beta:.4f}')
else:
    print('⚠ No EGFR graphs available; skipping fine-tuning')

# Save fine-tuned model
torch.save(vae.state_dict(), models_dir / 'egfr_finetuned_graphvae.pth')
print(f'\n✓ Fine-tuned VAE saved: {models_dir / "egfr_finetuned_graphvae.pth"}')


PHASE 2: Fine-tuning on EGFR (With Pocket Conditioning)


NameError: name 'get_working_device' is not defined

In [ ]:
# ===== PHASE 3: GENERATIVE SAMPLING =====

print('\n' + '='*60)
print('PHASE 3: Molecule Generation from Learned Latent Space')
print('='*60)

vae.eval()
n_samples = 8
print(f'\nGenerating {n_samples} molecules by sampling latent space...\n')

generated_graphs = []
with torch.no_grad():
    for i in range(n_samples):
        # Sample from standard normal (prior)
        z_sample = torch.randn(1, 64, device=device)
        
        # Decode to molecule graph
        node_features, edge_adj, edge_types = vae.decode(z_sample, pocket_embedding=None)
        
        generated_graphs.append({
            'idx': i + 1,
            'z_sample': z_sample[0].cpu().numpy(),
            'n_nodes': node_features.shape[1],
            'nodes_shape': node_features.shape,
            'edge_adj_shape': edge_adj.shape,
        })
        
        # Basic validation
        pos_nodes = (node_features.max(dim=-1).values > 0).sum().item()
        edge_density = (edge_adj > 0.5).float().mean().item()
        
        print(f'  Sample {i+1:2d}: generated {node_features.shape[1]} nodes, '\
              f'edge density: {edge_density:.2%}, positive nodes: {pos_nodes}')

print(f'\n✓ Generated {len(generated_graphs)} molecules')

# 3. Compare with unconditional baseline
print('\n' + '-'*60)
print('BASELINE: Random Latent Vectors (No Training)')
print('-'*60)

baseline_graphs = []
with torch.no_grad():
    for i in range(n_samples):
        z_random = torch.randn(1, 128, device=device)  # Different distribution for comparison
        try:
            node_features, _, _ = vae.decode(z_random, pocket_embedding=None)
            baseline_graphs.append(node_features.shape)
            print(f'  Baseline {i+1:2d}: {node_features.shape}')
        except Exception as e:
            print(f'  Baseline {i+1:2d}: Failed (expected—different latent dist)')
            baseline_graphs.append(None)

# 4. Save results
print('\n' + '='*60)
print('RESULTS & SUMMARY')
print('='*60)

results_dir = ROOT / 'results' / 'generated_mols'
results_dir.mkdir(parents=True, exist_ok=True)

summary = {
    'phase': 'VAE_Generative_Sampling',
    'n_generated': len(generated_graphs),
    'model_path': str(models_dir / 'zinc_pretrained_graphvae.pth'),
    'latent_dim': 64,
    'max_nodes': 30,
    'samples': [
        {
            'idx': g['idx'],
            'z_mean': float(np.mean(g['z_sample'])),
            'z_std': float(np.std(g['z_sample'])),
            'n_nodes': g['n_nodes'],
        }
        for g in generated_graphs
    ]
}

import json
with open(results_dir / 'vae_generation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nGenerated molecules: {summary["n_generated"]}')
print(f'Model saved to: {summary["model_path"]}')
print(f'Results saved to: {results_dir / "vae_generation_summary.json"}')
print(f'\n✅ VAE generative model complete!')


PHASE 3: Molecule Generation from Learned Latent Space

Generating 8 molecules by sampling latent space...

  Sample  1: generated 50 nodes, edge density: 0.00%, positive nodes: 50
  Sample  2: generated 50 nodes, edge density: 0.00%, positive nodes: 50
  Sample  3: generated 50 nodes, edge density: 0.00%, positive nodes: 50
  Sample  4: generated 50 nodes, edge density: 0.00%, positive nodes: 49
  Sample  5: generated 50 nodes, edge density: 0.00%, positive nodes: 49
  Sample  6: generated 50 nodes, edge density: 0.00%, positive nodes: 50
  Sample  7: generated 50 nodes, edge density: 0.00%, positive nodes: 50
  Sample  8: generated 50 nodes, edge density: 0.00%, positive nodes: 50

✓ Generated 8 molecules

------------------------------------------------------------
BASELINE: Random Latent Vectors (No Training)
------------------------------------------------------------
  Baseline  1: Failed (expected—different latent dist)
  Baseline  2: Failed (expected—different latent dist)
  B